# Silver Layer: Full-Overwrite Reference Tables
Cleans and loads the small reference and flat-file sources into Silver Delta tables.
These are reloaded in full each run (overwrite), which is simple and inherently idempotent.

**Tables:** customers, products, supplier_price_list, marketing_spend

In [0]:
from pyspark.sql import functions as F, Window

STORAGE_ACCOUNT = "atliqlakeab"
BRONZE = f"abfss://lakehouse@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze"

## Customers
Trim names, standardize city casing, cast signup_date, dedupe on customer_id.

In [0]:
customers = (
    spark.read.parquet(f"{BRONZE}/customers")
    .withColumn("customer_name", F.trim("customer_name"))
    .withColumn("city",          F.initcap(F.trim("city")))
    .withColumn("signup_date",   F.to_date("signup_date"))
    .dropDuplicates(["customer_id"])
    .filter(F.col("customer_id").isNotNull())
)
customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("atliq.silver.customers")
print("customers ->", customers.count())

## Products
Trim names, standardize category casing, cast unit_price to decimal, dedupe on product_id.

In [0]:
products = (
    spark.read.parquet(f"{BRONZE}/products")
    .withColumn("product_name", F.trim("product_name"))
    .withColumn("category",     F.initcap(F.trim("category")))
    .withColumn("unit_price",   F.col("unit_price").cast("decimal(10,2)"))
    .dropDuplicates(["product_id"])
    .filter(F.col("product_id").isNotNull())
)
products.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("atliq.silver.products")
print("products ->", products.count())

## Supplier Price List (from CSV)
Keep only the latest effective_date per product using a window function, so each product has one current supplier cost.

In [0]:
spl_raw = spark.read.parquet(f"{BRONZE}/supplier_price_list")
w = Window.partitionBy("product_id").orderBy(F.col("effective_date").desc())
supplier = (
    spl_raw
    .withColumn("supplier_cost",  F.col("supplier_cost").cast("decimal(10,2)"))
    .withColumn("effective_date", F.to_date("effective_date"))
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
    .filter(F.col("product_id").isNotNull())
)
supplier.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("atliq.silver.supplier_price_list")
print("supplier_price_list ->", supplier.count())

## Marketing Spend (from CSV)
Cast date and numeric columns, full overwrite.

In [0]:
mkt = (
    spark.read.parquet(f"{BRONZE}/marketing_spend")
    .withColumn("spend_date",   F.to_date("spend_date"))
    .withColumn("spend_amount", F.col("spend_amount").cast("decimal(12,2)"))
    .withColumn("clicks",       F.col("clicks").cast("int"))
)
mkt.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("atliq.silver.marketing_spend")
print("marketing_spend ->", mkt.count())